# 海辺の風は何時間先まで予測できるか
## ――機械学習とカオスから考える予測可能性（ハンズオン）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shogo-ishikawa/2026_3Q_M4Th4/blob/main/projects/coastal-wind-chaos/notebooks/coastal_wind_hands_on.ipynb)

（このリンクは、作品がリポジトリの main ブランチに取り込まれた後に使えます。取り込み前は、下の「0. 準備」の `BRANCH` を作業ブランチ名に変えてください。）

### この教材で学ぶこと

1. 風向きには0°と360°の境目があり、ふつうの数とは扱いが違うこと
2. 単純な予報（持続予報・気候値予報）を「ものさし」にして、機械学習の効果を評価すること
3. 季節・時刻の情報と、直近の観測情報の役割の違い
4. 予測時間を延ばすと、精度がどう変わるか
5. 予測が外れる原因（入力の不足、モデルの不足、観測誤差、カオス）を区別する考え方
6. 合成データで学んだ方法を、自分の地元の気象庁データに使う方法

### 2つの実験は別のもの

- **実験A：風向きの機械学習予測**　合成データ（またはあなたの地点の観測データ）で、与えた入力・モデル・学習期間・評価指標のもとでの**実用的な予測性能**を調べます。
- **実験B：ローレンツ系の初期値鋭敏性（カオス）**　簡単な方程式で、初期値のわずかな違いが広がる様子を調べます。

実験Aで誤差が増えることを見ても、それだけで「カオスの壁を見つけた」「この地点の物理的な予測限界は何時間だ」とは言えません。2つの実験は、考え方を比べるために並べています。

### 進め方

- **基礎編（1〜7）**：全員が上から順に実行し、図と表を読み取ります。
- **選択編（8〜11）**：興味のある部分を選び、設定を1つずつ変えて結果を説明します。

データは、南側に海がある**仮想観測点の合成データ**です。気象庁の観測値ではありません。図の文字は、どの環境でも文字化けしないよう英語にしています。

## 0. 準備

Google Colab では、次のセルがリポジトリを `/content/2026_3Q_M4Th4-（ブランチ名）/` に取得し、この作品のフォルダ `projects/coastal-wind-chaos/` を作業ディレクトリにします。ブランチごとに別のフォルダに取得するので、`BRANCH` を変えて実行し直しても、前に取得した内容と混ざりません。手元のパソコンでリポジトリの中から開いた場合は、取得せずにそのフォルダを使います。このNotebookの中の相対パス（`data/…` など）は、すべて作品フォルダが基準です。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/shogo-ishikawa/2026_3Q_M4Th4.git"
BRANCH = "main"  # 作品が main に取り込まれる前は、作業ブランチ名に変える
PROJECT = Path("projects/coastal-wind-chaos")
IN_COLAB = "google.colab" in sys.modules


def clone_in_colab():
    """Colab で BRANCH のリポジトリを取得し、作品フォルダを返す（ブランチごとに別のフォルダ）。"""
    clone = Path("/content") / ("2026_3Q_M4Th4-" + BRANCH.replace("/", "-"))
    if not clone.exists():  # 取得済みなら取得し直さない（書き換えたファイルを残すため）
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(clone)],
                       check=True)
    if not (clone / PROJECT).exists():
        raise FileNotFoundError(f"{BRANCH} ブランチに {PROJECT} がありません。"
                                "BRANCH を作品のあるブランチ名に変えて、このセルを実行し直してください。")
    return clone / PROJECT


def find_project_dir():
    """作品フォルダを探す。手元では、今のフォルダとその上のフォルダから探す。"""
    if IN_COLAB:
        return clone_in_colab()
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "src" / "coastal_wind_chaos").exists():
            return folder
        if (folder / PROJECT / "src" / "coastal_wind_chaos").exists():
            return folder / PROJECT
    raise FileNotFoundError("作品フォルダが見つかりません。リポジトリの中でこのNotebookを開いてください。")


PROJECT_DIR = find_project_dir()
os.chdir(PROJECT_DIR)
print("作業ディレクトリ:", Path.cwd())

In [ ]:
# Colab では、この作品のパッケージ（と足りない依存ライブラリ）を入れる。
# 手元では、README の手順（pip install -e .）で入れておく。
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
sys.path.insert(0, str(PROJECT_DIR / "src"))  # インストール直後でも読み込めるようにする

from coastal_wind_chaos import environment_info

environment_info(PROJECT_DIR)  # 使ったPython・ライブラリの版とコミットを記録しておく

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from coastal_wind_chaos import lorenz_demo, plotting, sample_data
from coastal_wind_chaos import wind_pipeline as wp

plotting.apply_style()
pd.set_option("display.precision", 1)

## 1. 合成データを作る

仮想観測点は「南側に海、北側に陸」という設定です。日中は南の海から吹く**海風**、夜は北の陸から吹く**陸風**になるようにしています。実際の海岸では、海の向きや地形によって風向が変わるので、どこでも同じ向きになるわけではありません。

合成データは、次の成分を**風の東向き成分 u と北向き成分 v ごとに足し合わせて**作ります（角度どうしは足しません）。

| 成分 | 内容 |
| --- | --- |
| 周期成分 | 季節風（冬は北西寄り、夏は南東寄り）、海風・陸風（日周期）、気温・気圧の季節変化と日変化 |
| 背景成分 | 数日かけてゆっくり変わる確率的な変動（高気圧・低気圧の通過の代わり） |
| 不規則成分 | 1時間前の状態と相関する確率的な変動（AR(1)過程） |
| 前線通過 | ランダムな時刻に来て、風向の急変・気温の低下を起こす（既定では、通過の12時間前から南西寄りの風が強まり、気圧が下がる「予兆」つき） |
| 観測 | 観測誤差、16方位への丸め、静穏（0.2 m/s以下）、まれな欠測 |

これは簡単な式と乱数で作った「現象のモデル」で、大気の流れを物理法則で計算する数値予報ではありません。乱数の種（`seed`）が同じなら、同じデータになります。

In [ ]:
DATA_PATH = Path("data/sample_coastal_amedas.csv")
TRUTH_PATH = Path("data/sample_coastal_truth.csv")  # 観測誤差のない「真の状態」（教材用）

settings = sample_data.generate_sample_csv(DATA_PATH, preset="default", truth_path=TRUTH_PATH, seed=2026)
settings

保存したCSVは、気象庁「過去の気象データ・ダウンロード」のファイルと同じ見出しの構造です（文字コードは Shift_JIS）。先頭の数行を見てみましょう。

In [ ]:
lines = DATA_PATH.read_bytes().decode("cp932").splitlines()
for line in lines[:8]:
    print(line[:110])

- 1行目：本物では「ダウンロードした時刻」です。合成データでは、合成データであることと、作ったときの設定を書いています。
- 3行目：地点名、4行目：要素名、5行目：「風速(m/s)」の下にある「風向」、6行目：品質情報・均質番号・現象なし情報
- 7行目から：1時間ごとのデータ

**時刻の意味**：日本標準時です。各行の時刻は、その行の値が表す1時間の終わりの時刻です。例えば「2023/1/1 1:00:00」の行の日照時間は、0時から1時までの合計です。風向・風速は時刻の直前10分間の平均、気温と気圧はその時刻の値です。1日は1時から始まり、24時は翌日の「00:00:00」と書かれます。

次に、同じ関数で読み込み、予測に使える形に整えます。

In [ ]:
raw = wp.read_jma_csv(DATA_PATH)
raw.head()

In [ ]:
weather = wp.prepare_weather(raw)  # 品質の確認、16方位→角度、静穏、u と v、1時間刻みへの整列
wp.summarise_weather(weather)

## 2. 風向は「ふつうの数」ではない

風向は、風が**吹いてくる**方位を北から時計回りに測った角度です（北 = 0°、東 = 90°、南 = 180°、西 = 270°）。北は0°でもあり360°でもあります。350°と10°は、数としては340離れていますが、向きとしては20°しか離れていません。

In [ ]:
print("ふつうの平均       :", np.mean([350, 10]), "度（南になってしまう）")
print("風向の平均         :", round(wp.circular_mean_deg([350, 10]), 1), "度（北）")
print("ふつうの差         :", abs(350 - 10), "度")
print("角度としての差     :", wp.angular_error(350, 10), "度")

そこで、風を東向きの成分 u と北向きの成分 v に分けて扱います。風向 θ、風速 S のとき

- u = −S × sin θ（西風なら u > 0）
- v = −S × cos θ（南風なら v > 0）

です（風は「吹いてくる向き」で表すので、向かう先はその反対になり、マイナスが付きます）。予測も u と v で行い、最後に角度に戻します。

In [ ]:
for label in ["北", "東", "南", "西", "北北西"]:
    degrees = wp.direction_label_to_degrees(label)
    u, v = wp.wind_to_uv(3.0, degrees)
    print(f"{label:>3}の風（{degrees:5.1f}°, 3 m/s） → u = {u:+.2f}, v = {v:+.2f} → 角度に戻すと {wp.uv_to_direction(u, v):5.1f}°")

**練習**　340°・10°・20° の3つの風向の平均を、ふつうの平均と `wp.circular_mean_deg` で比べてみましょう。

<details><summary>解答例</summary>

ふつうの平均は約123°（東南東）ですが、風向の平均は約3°（ほぼ北）です。3つとも北寄りなので、北が正しい答えです。

</details>

In [ ]:
# 練習：ここに書いて実行してみよう

## 3. データを眺める：海風と陸風

まず、夏と冬の1週間ずつを見ます。風向は点で描いています。線で結ぶと、北風のたびに0°と360°の間を縦に飛ぶ線になってしまうからです。

In [ ]:
plotting.plot_direction_timeseries(weather, "2024-07-01", days=7)
plt.show()
plotting.plot_direction_timeseries(weather, "2024-01-08", days=7)
plt.show()

次に、3年分をまとめて「何時に、どの方位から風が吹いたか」の割合を見ます（色が濃いほど多い）。北（N）は図の一番下と一番上の両方に近い位置です（N と NNW は隣どうし）。

In [ ]:
plotting.plot_direction_by_hour(weather)
plt.show()

In [ ]:
# 同じ内容を表で確かめる（行：16方位、列：時刻、値：%）
frequency = wp.direction_frequency_by_hour(weather)
frequency.index = wp.DIRECTION_LABELS
frequency.round(0)

**読み取り**：昼（12〜16時ごろ）は南〜南南東から、夜は北西〜北から吹くことが多く、夕方には西寄りを経て向きが時計回りに変わります。これはこの仮想地点に与えた設定どおりの海風・陸風です。季節風や数日規模の変動、前線が重なるため、毎日同じになるわけではありません。

## 4. 予報の「ものさし」：単純な予報

ここでは、ある時刻（**予測時刻**：予報を出す時刻）に、その何時間か後の時刻（**対象時刻**）の風向を予報します。機械学習が役に立つかどうかは、単純な予報と比べないと分かりません。ここでは2つの単純な予報を使います。

- **持続予報**：「今の風がそのまま続く」と予報する
- **気候値予報**：学習期間の「同じ月・同じ時刻の平均の風」を予報にする

評価は**最後の365日（テスト期間）**で行い、それより前の期間だけで学習します。ランダムに分けると、テスト期間の直前・直後の情報が学習に混ざってしまうので、時間で分けます。予測の対象時刻がテスト期間に入る行は、学習に使いません。

In [ ]:
test_start = wp.default_test_start(weather)
table6 = wp.make_forecast_table(weather, horizon=6)  # 6時間先を予測するための表
train6, test6 = wp.split_train_test(table6, test_start)
print("テスト期間の開始         :", test_start)
print("学習の行数 / テストの行数:", len(train6), "/", len(test6))
print("学習に使う正解の最後の時刻:", train6["target_time"].max())

In [ ]:
BASELINES = ["persistence", "climatology"]
forecasts6 = wp.forecast_one_horizon(weather, 6, BASELINES, test_start)
wp.score_forecasts(forecasts6, BASELINES)

`mae_deg` は平均絶対角度誤差（度）、`within_45_pct` は誤差が45°以内だった割合（%）、`n_test` は評価に使った時刻の数です。予測時刻か対象時刻の風向が決まらない時刻（静穏・欠測）と、使う入力が欠けている時刻は、すべての手法で同じように除くので、`n_test` はテストの行数より少し少なくなります。

もう1つの目安として、**正解と無関係に0〜360°から一様ランダムに方向を選ぶ予測**の誤差を計算してみます。期待値はちょうど90°です。

In [ ]:
random_errors = wp.random_direction_errors(forecasts6["target_direction_deg"].to_numpy(), seed=0)
print(f"独立な一様ランダム方向予測の平均誤差: {random_errors.mean():.1f}°（期待値 {wp.RANDOM_DIRECTION_EXPECTED_ERROR_DEG:.0f}°）")

この90°は「独立な一様ランダム方向予測の期待誤差」です。あらゆる予測の限界値ではありません。風向に偏りがあれば、気候値予報のように90°より小さくできることが多く、逆に、海風と陸風が入れ替わる時間差では、持続予報が90°より大きく外れることもあります。

## 5. 機械学習で予測する：季節・時刻の情報と、直近の観測

予測時刻 t（予報を出す時刻）の情報だけを使って、対象時刻 t + 6時間の風（u と v）を予測します。入力（特徴量）は2種類です。

- **季節・時刻**（calendar）：対象時刻の時刻と季節を sin・cos で表したもの（23時と0時が近いことを表せます）。暦は前もって分かるので、使ってもリークにはなりません。**リーク**とは、予測する時点ではまだ分からない未来の情報が、入力や学習に紛れ込むことです。リークがあると、実際には出せないほど良い成績になってしまいます。
- **直近の観測**（recent）：時刻 t とその1, 2, 3, 6, 12, 24時間前の u, v、t の気温・気圧・日照、3時間の気温変化・気圧変化。t より後の観測は使いません。

手法は2つです。

- **線形回帰**：入力に重みを掛けて足し合わせて予測します。ここでは、重みが大きくなりすぎないよう少し抑えた「リッジ回帰」を使います。
- **勾配ブースティング**：「入力がある値より大きいか」という質問を何段か重ねて予測する**決定木**を、それまでの誤差を小さくするように少しずつ足していく方法です。入力と答えの曲がった関係も表せます。

比較のため、**悪い例**として風向を「ふつうの数」のまま線形回帰する方法も入れます（時刻も0〜23のふつうの数として使います）。

In [ ]:
# 行の名前（issue_time）が予測時刻。target_u, target_v が6時間後の対象時刻の答え
print("予測時刻 → 対象時刻:", table6.index[200], "→", table6["target_time"].iloc[200])
columns = ["hour_sin", "hour_cos", "u_lag0", "v_lag0", "u_lag24", "pressure_change_3h", "target_u", "target_v"]
table6[columns].iloc[200:205].round(2)

In [ ]:
MODELS = [
    "persistence",
    "climatology",
    "linear:calendar",
    "linear:recent",
    "linear:calendar+recent",
    "boosting:calendar+recent",
    "degrees_as_numbers",
]
forecasts6 = wp.forecast_one_horizon(weather, 6, MODELS, test_start)
wp.score_forecasts(forecasts6, MODELS)

**読み取りのヒント**

- 季節・時刻だけの線形回帰は、気候値予報とほぼ同じくらいです。どちらも「その時刻のいつもの風」しか分かりません。
- 直近の観測を使うと、6時間先では大きく改善します。いまの風の状態がしばらく続くからです。
- 両方を使うと最もよくなります。勾配ブースティングは、このデータでは線形回帰とほぼ同じです。
- 角度を数値のまま回帰した悪い例は、ずっと悪くなります。

時刻別の誤差を見ると、悪い例が特に夜に外れることが分かります。夜は北寄りの陸風で、0°と360°の境目をまたぐからです。

In [ ]:
hourly6 = wp.error_by_hour(forecasts6, MODELS)
plotting.plot_error_by_hour(hourly6, "Error by hour of day, 6 hours ahead (test period)")
plt.show()

## 6. 予測時間を延ばす

1, 3, 6, 12, 24, 48時間先について、同じ評価を繰り返します（数十秒かかることがあります）。

In [ ]:
HORIZONS = [1, 3, 6, 12, 24, 48]
scores = wp.run_horizon_experiment(weather, HORIZONS, MODELS, test_start)
scores.pivot(index="model", columns="horizon_h", values="mae_deg").loc[MODELS].round(1)

In [ ]:
plotting.plot_error_vs_horizon(scores)
plt.show()

**読み取り**

- 直近の観測を使う予測は、予測時間が延びるほど誤差が大きくなり、気候値予報の誤差（約50°）に近づいていきます（48時間先でも、まだ少し小さい）。誤差が90°まで増えるわけではありません。
- 持続予報は12時間先で大きく外れ（海風と陸風が入れ替わる）、24時間先・48時間先では小さくなります。これは**日周期**のためで、同じ時刻には似た風が吹きやすいからです。カオス的な細かい情報が回復したわけではありません。
- ここで分かるのは、「この入力・このモデル・この学習期間・この評価指標」での**実用的な予測性能**です。この結果から、この地点の物理的な予測限界が何時間かを決めることはできません。

## 7. 基礎編のまとめ

- 風向は円の上の量なので、u・v（またはsin・cos）で扱い、角度の差で評価する。
- 単純な予報（持続予報・気候値予報）と比べて、はじめて機械学習の効果が分かる。
- 直近の観測は、予測時間が短いほどよく効き、延びると効果が小さくなる（このデータでは48時間先でも、季節・時刻だけの予測より誤差が小さい）。
- 季節・時刻の情報は、直近の観測に加えると6〜12時間先で特によく効く。24・48時間先では、予測時刻と対象時刻が同じ時刻になり、直近の観測が日周期の情報をすでに持っているので、上乗せの効果は小さい。
- 予測時間を延ばすと誤差は増える。持続予報の24・48時間先の誤差が12時間先より小さいのは日周期のためで、カオス的な情報が回復したわけではない。

ここから先は選択編です。興味のある節を選び、**設定を1つずつ変えて**、結果がどう変わったかを説明してみましょう。パッケージのコード（`.py` ファイル）を書き換えるときは、先に11節の最初の注意を読んでください。生成AIに質問するときは、例えば「`wp.make_forecast_table` の `shift(-horizon)` が何をしているか、具体的な時刻で説明して」のように、関数名と知りたいことをはっきり書くと答えを確かめやすくなります。AIの説明は、実際にコードを実行して確かめてください。

## 8. （選択）予測が外れる原因を区別する

予測が外れる原因には、例えば次のようなものがあります。

| 原因 | 確かめ方の例 |
| --- | --- |
| 入力の不足 | 入力を足したり減らしたりして、誤差が変わるか |
| モデルの不足 | 同じ入力のまま、モデルだけを変えて誤差が変わるか |
| 観測誤差 | 観測誤差のない「真の値」と比べたり、観測誤差の大きさを変えたりする |
| 確率的な変動 | 確率的な成分を入れたデータと入れないデータで比べる |
| カオス | 決定論的な系で初期値をわずかに変える（9節のローレンツ系） |

「1つの要因だけを変えて、誤差が変わるかを見る」のが基本です。合成データでは成分を自由に切り替えられ、真の状態も分かるので、この練習ができます。8節のセルは、前のセルで作った変数を使うので、8.1から順に実行してください。

### 8.1 入力の不足：気圧を入力から外す

In [ ]:
no_pressure = weather.drop(columns=["sea_level_pressure_hpa"])
pressure_scores = wp.run_scenario_experiment(
    {"with pressure": weather, "without pressure": no_pressure},
    [6, 12, 24], ["linear:calendar+recent", "boosting:calendar+recent"],
)
pressure_scores.pivot_table(index=["scenario", "model"], columns="horizon_h", values="mae_deg").round(1)

全体の平均では差は小さくなります。前線が来るのは数日に1回で、ほとんどの時間には気圧の情報が効かないからです。そこで、前線の前後だけで比べます。

真の状態のファイルには、前線の通過時刻が入っています（予測の入力には使わず、評価の分類だけに使います）。行の `[0, 3)` は「対象時刻が前線通過から3時間未満」、`[-3, 0)` は「通過の3時間前から直前」、列の `n` はその区間の時刻の数です。

In [ ]:
truth = sample_data.read_truth_csv(TRUTH_PATH)
FRONT_MODELS = ["persistence", "linear:calendar+recent", "boosting:calendar+recent"]
with_pressure = wp.forecast_one_horizon(weather, 6, FRONT_MODELS, test_start)
without_pressure = wp.forecast_one_horizon(no_pressure, 6, FRONT_MODELS, test_start)
fronts_with_pressure = sample_data.error_around_fronts(with_pressure, FRONT_MODELS, truth)
fronts_without_pressure = sample_data.error_around_fronts(without_pressure, FRONT_MODELS, truth)
pd.concat({"気圧あり": fronts_with_pressure, "気圧なし": fronts_without_pressure}, axis=1).round(0)

**読み取り**：気圧を外すと、前線通過の直後（`[0, 3)`、`[3, 6)`）で機械学習の誤差が大きくなります。持続予報は気圧を使わないので変わりません。全体の平均では小さな差でも、**どんなときに外れるか**に分けて調べると、入力の不足が見つかることがあります。

### 8.2 前線の予兆：単地点の過去の情報で急変を当てられるか

この合成データの前線には、通過の12時間前から南西寄りの風が強まり、気圧が下がる「予兆」があります。同じ乱数の種で、予兆だけをなくしたデータ（前線の時刻と強さは同じ）と比べます。

In [ ]:
settings_no_precursor = sample_data.make_settings("no_precursor", seed=2026)
truth_no_precursor = sample_data.simulate_truth(settings_no_precursor)
weather_no_precursor = wp.prepare_weather(sample_data.observe(truth_no_precursor, settings_no_precursor))
without_precursor = wp.forecast_one_horizon(weather_no_precursor, 6, FRONT_MODELS, test_start)
fronts_without_precursor = sample_data.error_around_fronts(without_precursor, FRONT_MODELS, truth_no_precursor)
pd.concat({"予兆あり": fronts_with_pressure, "予兆なし": fronts_without_precursor}, axis=1).round(0)

**読み取り**：予兆があるデータでは、前線通過の直後（`[0, 3)`、`[3, 6)`）の機械学習の6時間先予報の誤差が、予兆がないデータより小さくなります。気圧の低下など、前もって現れる変化があれば、単地点の過去の観測からある程度予測できます。一方、通過前（`[-12, -6)` から `[-3, 0)`）の誤差は、予兆があるデータの方が大きくなります。予兆そのものも風を南西寄りに変えるので、予兆が始まる前に出した予報は外れやすくなるからです。持続予報は、予兆があっても通過の直後に大きく外れたままです。

予兆のない外からのランダムな到来は、**その地点の過去の情報だけでは当てられません**。その場合は、ほかの地点の観測や数値予報など、別の入力が必要です。

### 8.3 モデルの不足：同じ入力でモデルだけを変える

6節の表で、同じ入力（季節・時刻＋直近の観測）の線形回帰と勾配ブースティングを比べてください。このデータでは全体の差は小さく、**外れる主な原因はモデルの形ではない**と考えられます。一方、

- 角度を数値のまま回帰した悪い例は、同じ情報を持っていても大きく外れます（表し方＝モデルの不足）。
- 8.1・8.2の前線の直後では、勾配ブースティングの方が線形回帰より誤差が小さく、予兆の情報をうまく使えています。

ただし、複雑なモデルがいつもよいとは限りません。季節・時刻だけを入力にした勾配ブースティング（`"boosting:calendar"`）を、学習期間とテスト期間の両方で評価してみます。`wp.forecast_training_period` は、学習に使った時刻を、そのモデル自身で予測します（学習に使った時刻なので、ふつうの評価には使いません）。

In [ ]:
OVERFIT_MODELS = ["climatology", "boosting:calendar"]
in_training = wp.forecast_training_period(weather, 6, OVERFIT_MODELS, test_start)
in_test = wp.forecast_one_horizon(weather, 6, OVERFIT_MODELS, test_start)
pd.DataFrame({
    "学習期間の誤差": wp.score_forecasts(in_training, OVERFIT_MODELS).set_index("model")["mae_deg"],
    "テスト期間の誤差": wp.score_forecasts(in_test, OVERFIT_MODELS).set_index("model")["mae_deg"],
})

**読み取り**：学習期間では勾配ブースティングの方が気候値予報より誤差が小さいのに、テスト期間では逆に大きくなります。柔軟なモデルが、学習期間（2年分）だけにあった偶然の変動まで覚えてしまった（**過学習**）と考えられます。気候値予報も学習期間の方が誤差が小さいのは、学習期間の平均そのものを予報にしているからです。

### 8.4 観測誤差：観測値と「真の風向」

観測値は16方位に丸められ、観測誤差も含みます。観測した風向と、真の風向の差を調べます。

In [ ]:
print(f"観測した風向と真の風向の差（平均）: {sample_data.observation_error_summary(weather, truth):.1f}°")
sample_data.errors_against_truth(forecasts6, ["persistence", "linear:calendar+recent"], truth)

In [ ]:
rows = []
for noise in [0.0, 0.3, 1.0]:
    noisy_settings = sample_data.make_settings("default", seed=2026, observation_noise_ms=noise)
    noisy_truth = sample_data.simulate_truth(noisy_settings)
    noisy_weather = wp.prepare_weather(sample_data.observe(noisy_truth, noisy_settings))
    noisy_forecasts = wp.forecast_one_horizon(noisy_weather, 1, ["linear:calendar+recent"], test_start)
    row = sample_data.errors_against_truth(noisy_forecasts, ["linear:calendar+recent"], noisy_truth)
    row.insert(0, "observation_noise_ms", noise)
    row["observation_vs_truth_deg"] = sample_data.observation_error_summary(noisy_weather, noisy_truth)
    rows.append(row)
pd.concat(rows, ignore_index=True)

**読み取り**：観測誤差が0でも、16方位（22.5°刻み）に丸めるだけで平均5〜6°ずれます。そのため、連続的な角度で予測する機械学習は、予測が完全に正しくても16方位の観測値とは平均5〜6°ずれます。一方、持続予報は16方位のまま予測するので、風向が変わらなければ誤差0°になり、その分だけ有利です。観測誤差を大きくすると、観測値に対する誤差（`mae_vs_observation_deg`）は大きく増えますが、真の風向に対する誤差（`mae_vs_truth_deg`）の増え方はそれより小さくなります。「外れた」と見える誤差の一部は、観測そのものの誤差です。実際の観測でも、風向計の向きのずれや周りの建物の影響などで誤差が生じることがあります。

### 8.5 確率的な変動：周期成分だけ と 周期成分＋AR変動

同じ手法を、成分の違う合成データで比べます。

In [ ]:
scenario_weather = {name: sample_data.generate_weather(name, seed=2026)
                    for name in ["periodic_only", "periodic_ar", "default"]}
SCENARIO_MODELS = ["persistence", "climatology", "linear:calendar+recent"]
scenario_scores = wp.run_scenario_experiment(scenario_weather, HORIZONS, SCENARIO_MODELS)
plotting.plot_scenario_comparison(scenario_scores, ["persistence", "linear:calendar+recent"])
plt.show()
scenario_scores.pivot_table(index=["scenario", "model"], columns="horizon_h", values="mae_deg").round(1)

**読み取り**

- **周期成分だけ**：気候値予報と機械学習の誤差は、どの予測時間でもほぼ同じ小ささ（約11°）で、残る誤差は主に観測誤差と16方位の丸めです。持続予報は12時間先で大きく外れますが（海風と陸風が入れ替わる）、24・48時間先ではほとんど外れません。ほぼ1日ごとに同じ風が繰り返すからです（季節の変化はゆっくりです）。
- **周期成分＋AR変動**：予測時間が延びると誤差が増え、半日ほどで気候値予報と同じくらいになって頭打ちになります。AR変動は「1時間前と相関する乱数」で、これから加わる新しい乱数は誰にも予測できないからです。これは**確率的な変動による予測の難しさで、カオスとは別のもの**です（AR(1)は安定な確率過程です）。
- 頭打ちになるまでの時間（ここでは半日ほど）は、データを作るときに選んだ `ar_phi`（1時間前との相関の強さ）や `ar_strength`（周期成分と比べた強さ）で決まります（改造課題2で確かめられます）。設定で変えられる値なので、実際の大気の予測限界を表すものではありません。

## 9. （選択）カオス：ローレンツ系の初期値鋭敏性

ローレンツ（1963）の方程式は、対流を大きく単純化した3変数の方程式です。

dx/dt = σ(y − x),　dy/dt = x(ρ − z) − y,　dz/dt = xy − βz　（σ = 10, ρ = 28, β = 8/3）

乱数は使わず、初期値が決まれば未来は1通りに決まります（決定論的）。それでも、初期値のわずかな違いが指数関数的に広がります。ここで使う時間は**ローレンツ系の無次元の時間**で、時間（hour）や日には換算しません。

In [ ]:
start_points = lorenz_demo.attractor_points(20, seed=0)
runs = lorenz_demo.twin_runs(start_points[0], perturbation_size=1e-6, n_steps=3000)
plotting.plot_twin_runs(runs)
plt.show()

始めは2本の軌道が重なっていますが、距離（下の図、対数目盛）はほぼ直線的に、つまり指数関数的に増え、やがて別々の動きになります。

距離がおよそ e<sup>λt</sup>（t は時間、λ は定数）のように広がるとき、λ を**最大リアプノフ指数**と呼びます。λ が正なら、小さな違いが指数関数的に広がります。次のセルでは、20個の出発点で距離の対数の傾き（成長率）を求めます。また、**Benettin法**（2本の軌道の距離を一定時間ごとに小さく戻しながら、広がる割合を平均する計算方法）でも λ を求めます。

In [ ]:
TOLERANCE = 1.0
times, distances = lorenz_demo.twin_distances(start_points, 1e-10, 3500)
growth_rate = lorenz_demo.fit_growth_rate(times, distances, 1e-8, 1e-1)
lyapunov = lorenz_demo.estimate_lyapunov_benettin(start_points[0], n_steps=20000)
print(f"距離の対数の傾き（成長率）       : {growth_rate:.2f}")
print(f"Benettin法による最大リアプノフ指数: {lyapunov:.2f}（文献値 約{lorenz_demo.REFERENCE_LYAPUNOV_EXPONENT}）")
plotting.plot_separation_growth(times, distances, TOLERANCE, growth_rate)
plt.show()

In [ ]:
predictability = lorenz_demo.predictability_table(start_points, [1e-10, 1e-8, 1e-6, 1e-4, 1e-2], TOLERANCE, n_steps=3500)
plotting.plot_predictability_times(predictability, TOLERANCE)
plt.show()
predictability

**読み取り**：初期誤差 δ が許容誤差 Δ に達するまでの時間は、およそ ln(Δ/δ) ÷ λ です。そのため、初期誤差を10分の1にしても、時間は ln(10) ÷ λ ≈ 2.3 ÷ 0.91 ≈ 2.5（無次元の時間）しか延びません。初期誤差を小さくする努力に比べて、延びる時間はわずかです。これがカオスによる予測の難しさです。

最後に、安定なAR(1)過程と比べます。同じ乱数を与えて初期値だけ変えた2本のAR(1)は、差がどんどん**縮み**ます。AR(1)の予測が外れるのは、これから加わる新しい乱数のためです。

In [ ]:
ar_twin = lorenz_demo.ar1_twin(phi=0.85, n_steps=48)
ar_error = lorenz_demo.ar1_forecast_error_std(phi=0.85, noise_std=1.0, n_steps=48)
plotting.plot_ar1_vs_lorenz(ar_twin, ar_error, times, distances)
plt.show()

**注意**

- ローレンツ系は大気そのものではありません。時間の単位も無次元です。「何時間で外れる」という数字は、時間の換算の仕方を決めればいくらでも作れてしまいます。
- この3変数のローレンツ系では、初期誤差を小さくすれば、予測できる時間は（ゆっくりとですが）いくらでも延ばせます。大気のように大小さまざまな規模の運動がある系では、初期誤差を小さくしても延ばせない限界があると考えられています（Lorenz, 1969）。
- 実験A（風向きの予測）で誤差が増えたのは、この合成データでは主に確率的な成分（AR変動・背景成分・前線）と入力の不足のためです。実験Aはカオスを示していません。
- 実際の大気の予測可能性は、数値予報モデルのアンサンブル実験などで研究されています。1地点の機械学習の誤差曲線から、その限界を決めることはできません。

## 10. （選択）自分の地元の気象庁データで試す

1. 気象庁「[過去の気象データ・ダウンロード](https://www.data.jma.go.jp/risk/obsdl/index.php)」で、風向・風速を観測している地点を選びます（気圧は気象台などでだけ観測しています）。
2. 「時別値」を選び、項目は「気温」「風向・風速」「日照時間」（気象台などなら「海面気圧」も）を選びます。
3. 期間は2〜3年を目安にします。一度にダウンロードできる量には上限があるので、多いときは1年ずつに分けます。学習には、テスト期間（既定では最後の365日）より前に、少なくとも30日（できれば1年以上）のデータが必要です。
4. 表示オプションは既定のまま（「すべて数値で格納」「日付リテラルで格納」、品質情報・均質番号を含める）でCSVをダウンロードします。
5. 下のセルを一度実行すると、アップロード先のフォルダ `data/raw/` ができ、その場所が表示されます（Colab では `/content/2026_3Q_M4Th4-main/projects/coastal-wind-chaos/data/raw/` のような場所です。`main` の部分は `BRANCH` の名前）。Colab では、左の「ファイル」でそのフォルダを開き、ファイルをアップロードします。
6. アップロードしたファイルを `MY_FILES` に書いて、もう一度実行します。

詳しくは作品の `data/README.md` を見てください。自動でダウンロードするプログラムは使わず、手作業でダウンロードしてください（気象庁は自動化ツールによる過度のアクセスを控えるよう求めています）。ダウンロードしたデータはGitに追加しないでください。

In [ ]:
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)  # アップロード先のフォルダを作る
print("アップロード先:", RAW_DIR.resolve())

MY_FILES = []    # 例：[RAW_DIR / "station_2023.csv", RAW_DIR / "station_2024.csv"]。1年ずつのファイルなら複数並べる
TEST_DAYS = 365  # 最後の何日をテスト期間にするか

if MY_FILES:
    my_weather = wp.load_weather(MY_FILES)
    display(wp.summarise_weather(my_weather))
    my_test_start = wp.default_test_start(my_weather, TEST_DAYS)
    my_scores = wp.run_horizon_experiment(my_weather, HORIZONS, MODELS, my_test_start)
    display(my_scores.pivot(index="model", columns="horizon_h", values="mae_deg").loc[MODELS].round(1))
    plotting.plot_direction_by_hour(my_weather)
    plt.show()
    plotting.plot_error_vs_horizon(my_scores)
    plt.show()
else:
    print("MY_FILES にファイルを書くと、同じ実験ができます。")

**実データで気をつけること**

- 結果は「その地点・その期間・その入力・そのモデル」での実用的な予測性能です。物理的な予測限界ではありません。
- 海風・陸風の向きは、海岸の向きや地形によって地点ごとに違います。風向計の高さや周りの建物も影響します。
- `summarise_weather` の「均質番号が変わった時刻」は、観測場所の移転や測器の変更の目安です。前後で値を単純に比べられないことがあります。
- アメダスの日照時間は、2021年3月2日から気象衛星のデータによる推計値です。
- 気象庁のデータは、後から修正されることがあります。ダウンロードした日を記録しておきましょう。
- 図や表に使うときは「出典：気象庁ホームページ（https://www.data.jma.go.jp/risk/obsdl/index.php）」のように出典を書き、加工した場合はそのことと加工した人を書きます。
- 自分で作った将来の予報を、他の人に向けて繰り返し公表すると、気象業務法の予報業務の許可が必要になることがあります。この教材では、過去のデータで学習と評価をするだけにしてください。

## 11. （選択）改造課題

### コードを書き換えるときの注意

- この作品のコードは、0節のセルが表示した作業ディレクトリの `src/coastal_wind_chaos/` にあります（Colab では `/content/2026_3Q_M4Th4-main/projects/coastal-wind-chaos/src/coastal_wind_chaos/` のような場所です）。Colab では、左の「ファイル」から `.py` ファイルを開いて書き換えられます。
- `.py` ファイルを書き換えても、実行中のカーネル（Pythonを実行している仕組み）は、前に読み込んだ内容を使い続けます。書き換えたら、Colab では「ランタイム → セッションを再起動」、手元の Jupyter では「Kernel → Restart」を選び、0節のセルから実行し直してください。
- Colab のランタイムを削除したり、しばらく使わずに接続が切れたりすると、`/content` のファイルは消えます。書き換えたファイルは、ダウンロードして保存しておきましょう。
- Notebookのセルに新しい関数を書いて試す方法なら、再起動は要りません。

### 課題

どれか1つを選び、**変える前の結果・変えたこと・変えた後の結果・理由**の順に説明してみましょう。

1. **直近の観測の時間**：`wp.forecast_one_horizon(weather, 6, MODELS, test_start, lags=(0,))` のように、使う「何時間前」を減らす。
2. **不規則成分の強さ**：`sample_data.generate_weather("periodic_ar", seed=2026, ar_strength=2.0)` で、AR変動を強くする・弱くする。
3. **前線の予兆の長さ**：`front_precursor_hours` を 0, 6, 12, 24 に変えて、8.2の表を比べる。
4. **海風の強さ**：`breeze_strength` を 0 や 2 にして、季節・時刻の情報の役割がどう変わるか調べる。
5. **悪い例を直す**：角度を数値のまま使う代わりに、sin・cos で表して予測する方法を考える（`wind_pipeline.py` の `fit_predict_degrees_as_numbers` を参考に。`wind_pipeline.py` を書き換えずに、Notebookのセルに新しい関数を書いて試すこともできます）。
6. **モデルの設定**：`wind_pipeline.py` の `make_regressor` で、勾配ブースティングの木の深さ `max_depth`（既定3）を `None`（制限なし）に変え、8.3と同じように、季節・時刻だけの場合（`"boosting:calendar"`）の学習期間とテスト期間の誤差を比べる。
7. **評価の仕方**：`within_45_pct`（45°以内の割合）で比べたり、風が弱い時刻を除いたりすると、手法の順位は変わるか。
8. **ローレンツ系**：`TOLERANCE` を変える。また、`lorenz_demo.RHO = 20.0` にしてから9節の `start_points` を作り直し、`lorenz_demo.estimate_lyapunov_benettin` と `lorenz_demo.predictability_table` を実行する（`lorenz_demo.fit_growth_rate` はエラーになります。なぜかも考えましょう。終わったら `lorenz_demo.RHO = 28.0` に戻す）。
9. **地元のデータ**：テスト期間の長さ（10節の `TEST_DAYS`）や季節を変えて評価する。

<details><summary>期待される観察と解答例</summary>

数値は、この教材の既定の合成データ（`seed=2026`）で確かめた例です。評価する時刻の選び方やライブラリの版で、少し変わることがあります。

1. 6時間先の誤差は少し大きくなります（季節・時刻＋直近の観測の線形回帰で約28.7°→約29.8°、直近の観測だけの線形回帰では約35°→約41°）。1時間前の値などから「風向が変わりつつあるか」が分かると、少し先の予測に役立つからです。なお、`lags` を変えると、評価に使う時刻（`n_test`）も変わるので、持続予報・気候値予報の値も少し動きます。
2. AR変動を強くすると、短い予測時間でも誤差が大きくなり、頭打ちになる誤差も大きくなります。頭打ちになるまでの時間も延びます（`ar_strength=0.5` では6時間先ごろ、`2.0` では12〜24時間先ごろ）。弱くすると逆になります。頭打ちまでの時間は、`ar_phi` だけでなく、周期成分と比べたAR変動の強さでも変わります。
3. 予兆が0だと、前線通過の直後の誤差は大きいままです。6時間や12時間の予兆があると小さくなります。ただし「長いほどよい」とは限らず、24時間のようにゆっくり変わる予兆では、いつ通過するかを見分けにくく、効果が小さくなることがあります。予兆があると、通過前の誤差は逆に大きくなることにも注意しましょう（8.2）。
4. 海風を0にすると昼夜の差がなくなり、季節・時刻の情報の効果は季節風の分だけになります。2にすると、季節・時刻だけの予測でも誤差が小さくなります。
5. 例えば目的変数を (sin θ, cos θ) にして線形回帰し、`np.arctan2` で角度に戻します。6時間先では、夜（対象時刻0〜3時）の誤差が約99°から約47°に大きく減ります。ただし、悪い例は時刻（0〜23）もふつうの数として使っているので、そのままでは昼（11〜14時）の誤差が約44°から約68°に増えます。時刻も sin・cos（`hour_sin`, `hour_cos`）で表すと全体で約45°、さらに予測時刻の風向も sin・cos で表すと約39°になります。
6. 深さの制限をなくすと、季節・時刻だけの勾配ブースティングは、学習期間の誤差がさらに小さくなる（約39°→約35°）のに、テスト期間の誤差はさらに大きくなり（約53°→約56°）、気候値予報（約50°）との差が広がります。学習期間には合うのにテスト期間には合わない、過学習の例です。
7. 例えば6時間先では、平均誤差では気候値予報（約50.4°）が持続予報（約56.2°）より良いのに、45°以内の割合では持続予報（約60.9%）が気候値予報（約56.3%）より高く、順位が入れ替わります。持続予報は、風向が変わらなければ誤差0°になる一方、海風と陸風が入れ替わると大きく外れるからです。指標によって見え方は変わりますが、角度の差で評価するという基本は同じです。弱い風の風向はもともと定まりにくいことにも注意しましょう。
8. 許容誤差を大きくすると時間は延びますが、初期誤差を10分の1にしたときに延びる時間（約2.5）はほぼ変わりません。ρ = 20 では、軌道はやがて2つの固定点（動きが止まる点）のどちらかに近づいて落ち着きます。Benettin法の値は約 −0.15 と負になり、`start_points` を作り直すと、2本の距離は時間とともに縮みます（`n_reached` はすべて0）。距離が指数関数的に増えないので、成長率を求める `fit_growth_rate` はエラーになります。なお、`start_points` を作り直さずに ρ = 28 のときの点から始めると、落ち着くまでの間は一時的にカオスのようにふるまう（過渡的カオス）ため、初期誤差が大きい組の一部は別々の固定点に行き着き、離れたままになります。
9. 地点や季節によって、どの情報が効くかが変わります。結果は、その地点・期間・入力・モデルでの性能として説明しましょう。

</details>

## 12. まとめ

- 風向は円の上の量です。u・v で扱い、角度の差で評価します。
- 単純な予報と比べることで、機械学習の効果を正しく評価できます。
- 予測が外れる原因は1つではありません。1つの要因だけを変えて比べると、入力の不足・モデルの不足・観測誤差・確率的な変動を区別できます。
- カオス（ローレンツ系）は、決定論的な系でも初期値のわずかな違いが広がる現象です。確率的な変動（AR過程）とは別のものです。
- 風向き予測の誤差曲線は、その条件での実用的な予測性能を表します。カオスの壁や物理的な予測限界を表すものではありません。